# Cloud Training Runner — Kaggle / Colab (Phase 2)

Generic runner for any milestone's `scripts/train/train_milestone_<x>.py`. Works on both
Kaggle Notebooks and Google Colab — set `PLATFORM` in the first cell and the rest adapts.

Full step-by-step context lives in `CLOUD_TRAINING.md` at the repo root — read that first if
anything here is unclear. This notebook is intentionally thin; it should not contain any
training logic itself (that lives in `scripts/train/train_milestone_<x>.py` so it stays
identical whether invoked from here, from Kaggle, or from Colab).

## Mandatory Preflight
Before training, the notebook **must** run:
```python
!python scripts/preflight/milestone_b_preflight.py --config configs/milestone_b.yaml
```
Exit code 1 = DO NOT START TRAINING. Fix the code first.

In [ ]:
# ---- CONFIGURE THIS CELL ----
PLATFORM = "kaggle"          # "kaggle" or "colab"
REPO_URL = "https://github.com/<you>/<repo>.git"
MILESTONE = "milestone_b"    # e.g. milestone_b, milestone_c, ... milestone_i
KAGGLE_DATASET_SLUG = "<your-kaggle-username>/<your-dataset-name>"   # only used if PLATFORM == kaggle
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/goal-conditioned-jepa"    # only used if PLATFORM == colab
RESUME_FROM = None            # e.g. "checkpoints/milestone_b/latest.pt", or None for a fresh run
GIT_USER_EMAIL = "you@example.com"
GIT_USER_NAME = "you"

# Optional: override data root for preflight (auto-discovered if not set)
DATA_ROOT = None

In [ ]:
# ---- GPU CHECK ----
!nvidia-smi

In [ ]:
# ---- MOUNT / ATTACH PERSISTENT STORAGE ----
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(f"{DRIVE_PROJECT_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_PROJECT_DIR}/data/metadit", exist_ok=True)
elif PLATFORM == "kaggle":
    # Attach the dataset via the Kaggle UI (Add Data) before running this cell.
    # This just confirms it's visible.
    !ls /kaggle/input/
else:
    raise ValueError("PLATFORM must be 'kaggle' or 'colab'")

In [ ]:
# ---- CLONE REPO + INSTALL DEPS ----
!git clone {REPO_URL} repo
%cd repo
!pip install -r requirements.txt -q

# Print environment info for provenance
import sys, torch, torchvision
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

# Git provenance
import subprocess
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()
print(f"Git commit: {commit}")
print(f"Git dirty: {bool(dirty)}")
if dirty:
    print(f"Dirty files:\n{dirty}")

In [ ]:
# ---- LINK DATA + CHECKPOINTS TO PERSISTENT STORAGE ----
# Use preflight discovery to find the exact dataset location
if PLATFORM == "kaggle":
    from pathlib import Path
    def discover_metadit_root():
        candidates = []
        for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
            if not base.exists():
                continue
            for p in base.rglob("train_set.mat"):
                root = p.parent.parent
                required = [
                    root / "split_data" / "train_set.mat",
                    root / "split_data" / "val_set.mat",
                    root / "split_data" / "test_set.mat",
                    root / "weights" / "spec_encoder.pth",
                    root / "weights" / "metadit-small.bin",
                ]
                if all(x.exists() for x in required):
                    candidates.append(root)
        candidates = list(dict.fromkeys(str(x.resolve()) for x in candidates))
        if len(candidates) != 1:
            raise RuntimeError(
                f"Expected exactly one valid MetaDiT dataset, found: {candidates}"
            )
        return Path(candidates[0])
    data_root = discover_metadit_root()
    repo_data = Path("data/metadit")
    if repo_data.exists() or repo_data.is_symlink():
        if repo_data.is_symlink() or repo_data.is_file():
            repo_data.unlink()
        else:
            import shutil
            shutil.rmtree(repo_data)
    repo_data.parent.mkdir(parents=True, exist_ok=True)
    repo_data.symlink_to(data_root, target_is_directory=True)
    print("MetaDiT data:", data_root)
    # /kaggle/working persists for the session and can be "Saved" as a notebook version
    !mkdir -p /kaggle/working/checkpoints
    !ln -sfn /kaggle/working/checkpoints checkpoints
elif PLATFORM == "colab":
    !ln -sfn {DRIVE_PROJECT_DIR}/data/metadit data/metadit
    !ln -sfn {DRIVE_PROJECT_DIR}/checkpoints checkpoints

In [ ]:
# ---- MANDATORY PREFLIGHT CHECK ----
print("Running mandatory preflight check...")
preflight_cmd = f"python scripts/preflight/milestone_b_preflight.py --config configs/{MILESTONE}.yaml"
if DATA_ROOT:
    preflight_cmd += f" --data-root {DATA_ROOT}"
import subprocess
result = subprocess.run(preflight_cmd, shell=True)
if result.returncode != 0:
    raise RuntimeError("PREFLIGHT FAILED. Exit code: " + str(result.returncode) + ". Fix the code before training.")
print("Preflight PASSED.")

In [ ]:
# ---- RUN TRAINING ----
resume_flag = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
!python scripts/train/train_{MILESTONE}.py --config configs/{MILESTONE}.yaml {resume_flag}

## Post-training verification (Phase 2 §13)
After training completes, verify the EXACT produced checkpoint:

```python
# Discover actual checkpoint path
from pathlib import Path
ckpts = sorted(Path("checkpoints/milestone_b").glob("*_latest.pt"))
if not ckpts:
    print("No existing checkpoints; starting fresh.")
else:
    for p in ckpts:
        print(p)
    CHECKPOINT = str(ckpts[-1])  # or choose after provenance inspection

!python scripts/diagnostics/checkpoint_provenance_audit.py
!python scripts/preflight/checkpoint_integrity_check.py --checkpoint {CHECKPOINT} --config configs/milestone_b.yaml
!python scripts/eval/eval_vicreg_sanity.py --checkpoint {CHECKPOINT} --config configs/milestone_b.yaml --device cuda:0
!python scripts/eval/physics_conditioning_audit.py --checkpoint {CHECKPOINT} --config configs/milestone_b.yaml --device cuda:0
```

Then compare final evaluation against in-loop validation. They must use the same validation/mask/metric implementation.

## Sync-back checklist (see `CLOUD_TRAINING.md` §4)
- [ ] Checkpoint saved to persistent storage (already true if using the symlinked path above)
- [ ] `checkpoints/<milestone>/REPORT.md` updated with metrics, done-criteria status, platform used
- [ ] Results pushed to GitHub (next cell) or manually copied back before closing this session

In [ ]:
# ---- OPTIONAL: PUSH RESULTS BACK TO GITHUB ----
# Requires a GitHub personal access token.
# On Kaggle: store it as a Kaggle Secret and load it here.
# On Colab: use `from google.colab import userdata; token = userdata.get('GITHUB_TOKEN')`
# then set the remote URL to include the token before pushing.
#
# !git config user.email "{GIT_USER_EMAIL}"
# !git config user.name "{GIT_USER_NAME}"
# !git add checkpoints/{MILESTONE}/
# !git commit -m "{MILESTONE}: cloud training run, see REPORT.md"
# !git push